In [45]:
import pandas as pd
import numpy as np
import pycountry_convert as pc

In [46]:
df = pd.read_csv( "data/LitLangMed_enriched.csv", dtype={43:str, 44:str, 45:str, 46:str} )
print( df.shape )
print( df.columns )

(7017, 60)
Index(['School', 'Folder', 'Document Name', 'Thesis Title', 'Thesis Author',
       'Citation ID', 'Citation Type', 'Citation Title', 'Citation Authors',
       'Citation Editors', 'Citation Year', 'Journal', 'Publisher',
       'Citation Source', 'Place', 'Series', 'Volume', 'Issue', 'Pages', 'DOI',
       'Reference URL', 'Note', '_is_duplicate_within_thesis', '_is_noise',
       '_noise_reason', '_citation_type_original', 'eligible_for_matching',
       '_ineligible_reason', 'Thesis Year', 'Citation Type (refined)',
       'Citation Author Count', 'Citation Is Single Author', 'ref_id',
       'openalex_status', 'openalex_affiliations', 'openalex_countries',
       'openalex_author_breakdown', 'openalex_author_country_pairs',
       'openalex_matched_author_count', 'openalex_main_author_matched',
       'openalex_distinct_author_countries',
       'openalex_is_international_collab', 'pyorcid_status',
       'pyorcid_affiliations', 'pyorcid_countries', 'pyorcid_author_break

In [47]:
# Drop rows with nan authors
df = df.dropna(subset=['Citation Authors'])
len(df)

6428

In [48]:
df[['Citation Authors', 'openalex_countries']]

,Citation Authors,openalex_countries
0,S Abbott,NaN
1,T Adorno; M Horkheimer,NaN
2,L Adams,NaN
3,R Addlakha; J Price; S Heidari,CHE
4,A Adut,USA
...,...,...
7012,E Zhuravskaya; M Petrova; R Enikolopov,ESP; FRA; RUS
7013,C Ziegler,USA
7014,A Zubiaga; H Ji,USA
7015,Samuel Yours Sincerely; Olaniran,NaN


In [49]:
def get_continent_name(country_code):
    try:
        # Convert 3-letter code to 2-letter code if necessary
        if len(country_code) == 3:
            country_code = pc.country_alpha3_to_country_alpha2(country_code)
            
        # Get the 2-letter continent code (e.g., 'AF', 'NA', 'EU')
        continent_code = pc.country_alpha2_to_continent_code(country_code.upper())
        
        # Map the continent code to its full name
        continent_name = pc.convert_continent_code_to_continent_name(continent_code)
        return continent_name
    except KeyError:
        return "Unknown"

In [62]:
continents = []

for index, row in df.iterrows():
    
    if row['openalex_countries'] is np.nan:
        continents.append(np.nan)
        continue
    
    author_continents = set()
    
    for country in row['openalex_countries'].split(';'):
        country = country.strip()
        author_continents.add( get_continent_name( country ) )

    if 'Africa' in author_continents:
        continents.append( 'Africa' )
    elif len(author_continents) == 1:
        continents.append( author_continents.pop() )
    else:
        continents.append( 'Multiple non-African' )

df['continent'] = continents

In [63]:
df[['openalex_countries', 'continent']]

,openalex_countries,continent
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,CHE,Europe
4,USA,North America
...,...,...
7012,ESP; FRA; RUS,Europe
7013,USA,North America
7014,USA,North America
7015,NaN,NaN
